In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, SshSource, SourceType, Std, DataInstanceLibrary, WorkflowTask
from local.constants import WORKSPACE_ROOT

dtypes, containers, transforms = Std()

# agent_home = Source.FromLocal(Path("./cache/local_home").resolve())
agent_home = SshSource("fir", Path("/scratch/phyberos/metasmith")).AsSource()
smith = Agent(
    home = agent_home,
)
# smith.Deploy()

In [ ]:
# dtypes.types

In [ ]:
# inputs.SaveAs(SshSource("fir", Path("/home/phyberos/project-rpp/cyanoverse/main/logistics/interleave_test.xgdb")).AsSource())

In [ ]:
inputs = DataInstanceLibrary.LoadFrom(
    src=SshSource("fir", Path("/home/phyberos/project-rpp/cyanoverse/main/logistics/interleave_test.xgdb")).AsSource(),
    dest=Path("./cache/read_interleave.remote.xgdb").resolve(),
    as_image=True,
    on_exist="skip"
)

inputs = DataInstanceLibrary.Load("./cache/read_interleave.xgdb")
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

In [ ]:
task = smith.GenerateWorkflow(
    given      = [containers, inputs],
    transforms = [transforms],
    targets    = [
        dtypes["short_reads"],
        # dtypes["per_bp_coverage"],
    ],
)

for step in [s for p in task.plans for s in p.steps]:
    print(step.order, step.transform.name)
# task.plans[0].RenderDAG("./cache/dag")

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
task_key = task.GetKey()
task_key

In [ ]:
smith.RunWorkflow(task_key)

In [ ]:
smith.CheckWorkflow(task_key)

In [ ]:
results = DataInstanceLibrary.Load(WORKSPACE_ROOT/"main/transforms/cache/local_home/runs/FLAaLASw/results/2025-10-20_16-08-22")

In [ ]:
for p, n, t in results.Iterate():
    print(p, n, t)